# m46 lab · Trustworthy AI: explanations, uncertainty, fairness, unfamiliar inputs and attacks

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You explain a beat classifier with integrated gradients and check whether the explanation depends on the model at
all; wrap a rhythm classifier in conformal prediction sets and see which class loses coverage; measure error
rates and predictive values in two patient groups; flag recordings the model was not built for; and find how
small a perturbation flips a decision.

| Part | What you do | Data |
| --- | --- | --- |
| A | Integrated gradients and the model-randomization check (eq-m46-ig) | synthetic beats |
| B | Conformal prediction sets for three rhythm classes (eq-m46-conformal) | synthetic ECGs |
| C | Error rates and predictive values in two groups (eq-m46-fpr) | synthetic ECGs |
| D | Unfamiliar inputs: a Mahalanobis score and abstention (eq-m46-mahalanobis) | synthetic ECGs |
| E | How small an attack flips a decision (eq-m46-fgsm) | synthetic beats |
| F | Subgroups and conformal coverage on real recordings | Challenge 2017 and PTB-XL (download) |
| G | Optional: integrated gradients with PyTorch autograd | synthetic beats |

All synthetic values (wave sizes, noise, prevalences, group mixes) are illustrative. Each part ends with
**checks**; the lab runs in under a minute (plus downloads in part F).

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import aiecg as ai
from epml import evidence as ev
from epml import foundation as fm
from epml import trust as tr
from epml.ml import auc_rank
from epml.retrieval import wilson_interval

FS = 100.0

def records(n, seed, af_share, ectopy_share, noise=0.03):
    """Synthetic 10-s single-lead ECGs; RR features from detected R peaks (m37). Returns features,
    AF labels (0/1), rhythm names."""
    r = np.random.default_rng(seed)
    F, y, kind = [], [], []
    for _ in range(n):
        u = r.random()
        rh = "af" if u < af_share else ("ectopy" if u < af_share + ectopy_share else "sinus")
        x, _ = ai.synth_ecg(rh, FS, 10.0, r.uniform(55, 110), r.uniform(0.6, 1.4), noise_mv=noise, rng=r)
        pk = ai.detect_r_peaks(x, FS)
        F.append(ev.rr_feature_rows([x], [pk], FS, extra=True)[0]); y.append(rh == "af"); kind.append(rh)
    return np.array(F), np.array(y, int), np.array(kind)

## Part A · Integrated gradients, and a check that the explanation depends on the model

A small neural network decides from a single beat (2,000 synthetic patients, 250 samples per second) whether the
QRS complex is wide. Integrated gradients (Sundararajan et al. 2017) attribute the output to each sample of the
beat, relative to a flat-line baseline; the attributions add up to the change in output (eq-m46-ig). Then the
model-randomization test of Adebayo et al. (2018): give the same architecture random weights and see whether the
explanation changes.

In [ ]:
rng = np.random.default_rng(1)
P = fm.patient_params(1500, rng)
X = fm.recordings(P, np.arange(1500), rng)
y_wide = (P["width"] > 1.4).astype(float)
mu, sd = X.mean(axis=0), X.std(axis=0) + 1e-9
Z = (X - mu) / sd
mlp = tr.TinyMLP(Z.shape[1], 16, seed=0).fit(Z[:1000], y_wide[:1000], lr=0.5, epochs=600)
auc_mlp = auc_rank(mlp.prob(Z[1000:]), y_wide[1000:])
i = 1000 + int(np.argmax(y_wide[1000:] * mlp.prob(Z[1000:])))     # a confidently wide beat
x, base = Z[i], (np.zeros(Z.shape[1]) - mu) / sd                    # flat line, in the model's units
ig = tr.integrated_gradients(mlp.grad_input, x, base, steps=128)
ig_rand = tr.integrated_gradients(tr.randomized_copy(mlp, seed=3).grad_input, x, base, steps=128)
gap = ig.sum() - (mlp.prob(x)[0] - mlp.prob(base)[0])
rho = tr.rank_correlation(np.abs(ig), np.abs(ig_rand))

t_ms = fm.BEAT_T * 1000
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8, 4.5), sharex=True)
a1.plot(t_ms, X[i]); a1.set_ylabel("beat (mV)")
a2.plot(t_ms, ig, label="trained model"); a2.plot(t_ms, ig_rand, alpha=0.7, label="random weights")
a2.set_xlabel("time from R peak (ms)"); a2.set_ylabel("attribution"); a2.legend(); plt.show()
qrs = np.abs(t_ms) <= 60
print(f"Test AUC {auc_mlp:.3f}. Completeness gap {gap:.1e}. Share of |attribution| within 60 ms of the R peak: "
      f"trained {np.abs(ig[qrs]).sum() / np.abs(ig).sum():.2f}, random {np.abs(ig_rand[qrs]).sum() / np.abs(ig_rand).sum():.2f}. "
      f"Rank correlation of |attributions|, trained vs random: {rho:.2f}.")

In [ ]:
# Checks
assert auc_mlp > 0.9 and abs(gap) < 1e-3
assert np.abs(ig[qrs]).sum() / np.abs(ig).sum() > np.abs(ig_rand[qrs]).sum() / np.abs(ig_rand).sum()
assert rho < 0.8                                   # the explanation does depend on the trained weights

The random model's attributions are not unrelated to the trained model's: integrated gradients multiply by
$x - x'$, so the shape of the beat leaks into any model's explanation. That is why the randomization test is a
minimum check, not a proof that an explanation is right.

## Part B · Conformal prediction sets for three rhythm classes

A multinomial logistic regression on RR features (coefficient of variation, normalized RMSSD, share of regular
beats) labels 10-s recordings as sinus rhythm, AF or sinus rhythm with frequent early beats. Split conformal
prediction (eq-m46-conformal; Angelopoulos and Bates 2021) uses 900 calibration recordings to turn its
probabilities into sets that contain the true class at least 90% of the time, on average over recordings.

In [ ]:
from sklearn.linear_model import LogisticRegression

def three_class(n, seed):
    F, _, kind = records(n, seed, 1 / 3, 1 / 3)
    return F, np.array([{"sinus": 0, "af": 1, "ectopy": 2}[k] for k in kind])

Ftr, ytr = three_class(900, 11); Fcal, ycal = three_class(900, 12); Fte, yte = three_class(900, 13)
clf = LogisticRegression(max_iter=3000).fit(Ftr, ytr)
Pcal, Pte = clf.predict_proba(Fcal), clf.predict_proba(Fte)
names = ["sinus", "AF", "early beats"]
res = {}
for by_class in (False, True):
    S = tr.conformal_sets(Pcal, ycal, Pte, 0.1, by_class=by_class)
    res[by_class] = {"overall": tr.coverage(S, yte),
                     "by class": [tr.coverage(S[yte == k], yte[yte == k]) for k in range(3)],
                     "size": S.sum(axis=1).mean(), "empty": (S.sum(axis=1) == 0).mean()}
print(f"Accuracy of the top class: {(Pte.argmax(axis=1) == yte).mean():.3f}")
for by_class, r in res.items():
    label = "one threshold per class" if by_class else "one threshold for all"
    print(f"{label:24s} coverage {r['overall']:.3f}; " + ", ".join(f"{n} {c:.3f}" for n, c in zip(names, r["by class"]))
          + f"; mean set size {r['size']:.2f}; empty sets {r['empty']:.3f}")

In [ ]:
# Checks
assert res[False]["overall"] >= 0.88                     # marginal guarantee (up to sampling noise)
assert min(res[False]["by class"]) < 0.86                # but one class is under-covered
assert min(res[True]["by class"]) > min(res[False]["by class"])

The guarantee is *marginal*: averaged over recordings, not within each class or patient group. One threshold per
class restores coverage in each class, at the cost of larger sets. Empty sets are information too: no class
looked plausible, which is what an unfamiliar recording often looks like (part D).

## Part C · Error rates and predictive values in two groups

An AF detector (logistic regression on the same RR features) is trained on one population and applied to two
groups: group A with few patients in AF or with early beats, group B with more of both (think of an older clinic
population; the mixes are illustrative). With different prevalences, equal sensitivity and false-positive rates
give different predictive values, and vice versa: from the definitions, $\mathrm{FPR} = \frac{\pi}{1-\pi}
\frac{1-\mathrm{PPV}}{\mathrm{PPV}}\mathrm{TPR}$ (eq-m46-fpr; Chouldechova 2017).

In [ ]:
Ftr2, ytr2, _ = records(800, 1, 0.3, 0.05)
detector = ev.fit_detector(Ftr2[:, :2], ytr2)                     # two RR features, as in m37
FA, yA, kA = records(1000, 2, 0.05, 0.05)
FB, yB, kB = records(1000, 3, 0.25, 0.25)
groups = np.array(["A"] * 1000 + ["B"] * 1000)
yy = np.r_[yA, yB]
pp = detector(np.r_[FA, FB][:, :2])
rates = tr.group_rates(yy, pp >= 0.5, groups)
for g, r in rates.items():
    print(f"group {g}: prevalence {r['prevalence']:.3f}, TPR {r['TPR']:.3f}, FPR {r['FPR']:.3f}, PPV {r['PPV']:.3f}")
need = tr.fpr_from_ppv(rates["B"]["prevalence"], rates["A"]["PPV"], rates["B"]["TPR"])
print(f"For group B to reach group A's PPV at its own TPR, its FPR would need to be {need:.3f} instead of {rates['B']['FPR']:.3f}.")
fpr_early = np.mean(pp[1000:][kB == "ectopy"] >= 0.5)
print(f"False positives in group B come mostly from early beats: FPR among those recordings {fpr_early:.2f}.")

In [ ]:
# Checks
assert rates["B"]["FPR"] > 3 * rates["A"]["FPR"]
for r in rates.values():
    assert abs(tr.fpr_from_ppv(r["prevalence"], r["PPV"], r["TPR"]) - r["FPR"]) < 1e-9

Which gap matters depends on the decision: a missed AF (TPR) or an unnecessary work-up (FPR, PPV). No single
number makes a model fair; report all of them per group, with intervals, and look for the cause, here the early
beats.

## Part D · Unfamiliar inputs: score them, and abstain

Recordings with much more noise (0.25 mV instead of 0.03 mV, illustrative) make the R-peak detector miss and
invent beats. A Mahalanobis distance on the features (eq-m46-mahalanobis; Lee et al. 2018 used it on network
features) flags inputs unlike the training data; the chi-square threshold sets a 1% false-alarm rate on familiar
data.

In [ ]:
Fn, yn, _ = records(400, 4, 0.3, 0.05, noise=0.25)
md_ = tr.Mahalanobis(Ftr2)
thr = md_.threshold(0.01)
s_in, s_out = md_.score(FA[:400]), md_.score(Fn)
ood_auc = auc_rank(np.r_[s_in, s_out], np.r_[np.zeros(400), np.ones(400)])
acc = lambda F, y: float(((detector(F[:, :2]) >= 0.5) == y).mean())
keep = s_out <= thr
print(f"OOD score AUROC {ood_auc:.3f}; flagged: familiar {np.mean(s_in > thr):.3f}, noisy {np.mean(s_out > thr):.3f}")
print(f"Accuracy: familiar {acc(FA[:400], yA[:400]):.3f}, noisy {acc(Fn, yn):.3f}, noisy but not flagged {acc(Fn[keep], yn[keep]):.3f}")

In [ ]:
# Checks
assert ood_auc > 0.75 and np.mean(s_in > thr) < 0.05
assert acc(Fn[keep], yn[keep]) > acc(Fn, yn)

## Part E · How small an attack flips a decision?

For a linear score $z = \mathbf w^\top \mathbf x + b$ on the raw beat (mV samples), changing every sample by at
most $\varepsilon$ changes $z$ by at most $\varepsilon\lVert\mathbf w\rVert_1$ (eq-m46-fgsm; Goodfellow et al.
2015), so $\lvert z\rvert / \lVert\mathbf w\rVert_1$ is the smallest such perturbation that flips the decision.
Han et al. (2020) built smoothed attacks on an ECG network that experts could not see; a moving-average smoothed
direction is our crude stand-in.

In [ ]:
w_all = fm.fit_logistic(X[:1000], y_wide[:1000], lam=1.0)
w, b = w_all[:-1], w_all[-1]
z = X[1000:] @ w + b
eps = np.array([tr.eps_to_flip_linear(w, b, xx) for xx in X[1000:]]) * 1000     # microvolts
d = tr.smooth(np.sign(w), 9); d /= np.abs(d).max()
eps_smooth = np.abs(z) / np.abs(w @ d) * 1000
print(f"Raw-beat model AUC {auc_rank(z, y_wide[1000:]):.3f}")
print(f"Median perturbation that flips the decision: {np.median(eps):.0f} uV (sign pattern), {np.median(eps_smooth):.0f} uV (smoothed); "
      f"the synthetic recordings already carry 40 uV of noise.")
j = int(np.argmin(np.abs(eps - np.median(eps))))
xa = X[1000 + j] - np.sign(z[j]) * eps[j] / 1000 * 1.01 * np.sign(w)
print("Decision before and after:", z[j] > 0, (xa @ w + b) > 0)

In [ ]:
# Checks
assert (xa @ w + b > 0) != (z[j] > 0)
assert np.median(eps_smooth) > np.median(eps)

## Part F · Subgroups and conformal coverage on real recordings

The m37 RR detector is trained on the PhysioNet/CinC Challenge 2017 (Clifford et al. 2017; Open Data Commons
Attribution v1.0) and tested on lead I of PTB-XL 1.0.3, fold 10 (Wagner et al. 2020; Creative Commons Attribution
4.0), sinus rhythm (SR) against AF. Sensitivity and specificity are reported by the sex code and by age group
(under 65, 65 and over; the cut is illustrative) from PTB-XL's metadata, with Wilson intervals (m43). Conformal
sets are calibrated on a Challenge hold-out and tested on PTB-XL: a different database, so the exchangeability the
guarantee needs may not hold. Without internet access this part is skipped.

In [ ]:
from scipy.signal import resample_poly

from epml.data import DataUnavailable, load_challenge2017, load_ptbxl_lead, ptbxl_metadata

try:
    sig, lab, _ = load_challenge2017(n_per_class=150, classes=("N", "A"))
    ext, ext_lab, ext_ids, _ = load_ptbxl_lead(n_per_class=60, codes=("SR", "AFIB"), fold=10)
    meta = ptbxl_metadata(ext_ids)
    REAL = True
except DataUnavailable as err:
    print("Part F skipped:", err)
    REAL = False

In [ ]:
if REAL:
    def rr_rows(signals):
        return ev.rr_feature_rows(signals, [ai.detect_r_peaks(np.asarray(s, float), FS) for s in signals], FS)

    xs = [resample_poly(np.asarray(s, float), 1, 3)[:1000] for s in sig]
    keep_c = [k for k, s in enumerate(xs) if len(s) == 1000]
    Fc = rr_rows([xs[k] for k in keep_c]); yc = np.array([lab[k] == "A" for k in keep_c], int)
    Fp = rr_rows([np.asarray(s)[:1000] for s in ext]); yp = np.array([l == "AFIB" for l in ext_lab], int)
    r = np.random.default_rng(7)
    hold = r.random(len(yc)) < 0.4
    det_real = ev.fit_detector(Fc[~hold], yc[~hold])
    p_ext = det_real(Fp)
    sex = np.array([meta[int(e)]["sex"] for e in ext_ids])
    age = np.array([meta[int(e)]["age"] for e in ext_ids])
    subgroups = {"sex code 0": sex == 0, "sex code 1": sex == 1, "age < 65": age < 65, "age >= 65": age >= 65}
    print(f"{'subgroup':12s} {'n':>4s} {'AF':>4s}  sensitivity (95% CI)      specificity (95% CI)")
    sub_rows = {}
    for name, m in subgroups.items():
        pos, neg = (yp == 1) & m, (yp == 0) & m
        tp, tn = int(np.sum(p_ext[pos] >= 0.5)), int(np.sum(p_ext[neg] < 0.5))
        sub_rows[name] = (wilson_interval(tp, pos.sum()), wilson_interval(tn, neg.sum()))
        show = lambda k, n, ci: f"{k / n:.2f} ({ci[0]:.2f}-{ci[1]:.2f})" if n else "no cases     "
        print(f"{name:12s} {m.sum():4d} {pos.sum():4d}  {show(tp, pos.sum(), sub_rows[name][0])}          "
              f"{show(tn, neg.sum(), sub_rows[name][1])}")
    P_hold = np.c_[1 - det_real(Fc[hold]), det_real(Fc[hold])]
    P_ext = np.c_[1 - p_ext, p_ext]
    S_ext = tr.conformal_sets(P_hold, yc[hold], P_ext, 0.1)
    cov_ext = tr.coverage(S_ext, yp)
    print(f"Conformal coverage on PTB-XL (target 0.90, calibrated on Challenge 2017): {cov_ext:.3f}; "
          + ", ".join(f"{n} {tr.coverage(S_ext[m], yp[m]):.2f}" for n, m in subgroups.items() if m.any()))

In [ ]:
# Checks
if REAL:
    assert 0 <= cov_ext <= 1 and len(sub_rows) == 4

With 120 PTB-XL recordings split four ways, each subgroup holds only a few dozen and the intervals are wide: subgroup analyses need planned sample sizes
(m45), not just a table. Coverage below 0.90 on PTB-XL would not be a failure of conformal prediction but of its
assumption: calibrate on data from the population where the sets will be used.

## Part G · Optional: integrated gradients with PyTorch autograd

The same check of eq-m46-ig with automatic differentiation on a small 1-D convolutional network. Runs only if
PyTorch is installed.

In [ ]:
try:
    import torch
except ImportError:
    torch = None
    if os.environ.get("EPML_REQUIRE_TORCH") == "1":
        raise
    print("PyTorch is not installed: part G skipped.")

if torch is not None:
    torch.manual_seed(0)
    net = torch.nn.Sequential(torch.nn.Conv1d(1, 8, 9, padding=4), torch.nn.Tanh(), torch.nn.AdaptiveAvgPool1d(8),
                              torch.nn.Flatten(), torch.nn.Linear(64, 1))
    opt = torch.optim.Adam(net.parameters(), lr=3e-3)
    Xt = torch.tensor(Z[:1000], dtype=torch.float32).unsqueeze(1); yt = torch.tensor(y_wide[:1000], dtype=torch.float32)
    for _ in range(300):
        opt.zero_grad()
        loss = torch.nn.functional.binary_cross_entropy_with_logits(net(Xt).squeeze(1), yt)
        loss.backward(); opt.step()
    F = lambda v: torch.sigmoid(net(v)).squeeze()
    xt = torch.tensor(x, dtype=torch.float32).view(1, 1, -1); bt = torch.tensor(base, dtype=torch.float32).view(1, 1, -1)
    grads = []
    for a in (np.arange(128) + 0.5) / 128:
        pt = (bt + float(a) * (xt - bt)).requires_grad_(True)
        g, = torch.autograd.grad(F(pt), pt)
        grads.append(g)
    ig_t = ((xt - bt) * torch.stack(grads).mean(0)).squeeze().detach().numpy()
    gap_t = ig_t.sum() - (F(xt) - F(bt)).item()
    print(f"CNN: completeness gap {gap_t:.1e} (output change {(F(xt) - F(bt)).item():.3f})")

In [ ]:
# Checks
if torch is not None:
    assert abs(gap_t) < 0.02

## Where to go next

- In part A, use a baseline that is the average beat instead of a flat line. How do the attributions change?
- In part B, calibrate on data with a different class mix from the test data. What happens to coverage?
- In part C, choose a separate threshold for each group to equalize sensitivity. What happens to PPV?
- In part E, constrain the perturbation to frequencies below 40 Hz. How large must it be now?